# Apprentissage Non Supervisé - Startups Américaines

**Dataset : 50_Startups.csv**

50 startups américaines décrites par leurs dépenses (R&D, Administration, Marketing) et leur bénéfice annuel.

---

## Configuration et Imports

In [ ]:
from utils_ans import *
setup_environment()

---
# I. Réduction de dimensions et Visualisation des données
---

## 1. Importation du jeu de données

In [ ]:
data = load_data('./data/50_Startups.csv')

In [ ]:
data.describe()

In [ ]:
feature_cols = ['Depenses R&D', 'Depenses Administration', 'Depenses Marketing Spend', 'Benefice']
X, labels, feature_names = prepare_data(data, feature_cols=feature_cols, label_col='Id')

## 2. Analyse en Composantes Principales (ACP)

In [ ]:
X_scaled, X_pca, pca, scaler = perform_pca(X)

In [ ]:
cumulative = print_variance_explained(pca)

In [ ]:
plot_variance(pca)

### Réponse : Nombre d'axes à retenir

- PC1 : ~65% de variance
- PC1+PC2 : ~87% de variance

**Conclusion : 2 axes suffisent** pour une bonne représentation (>85% de variance).

In [ ]:
loadings_df = get_loadings(pca, feature_names)

### Interprétation des axes

**PC1 - "Taille/Performance"** : corrélé avec R&D, Marketing et Bénéfice

**PC2 - "Stratégie d'investissement"** : opposition R&D/Marketing vs Administration

In [ ]:
plot_correlation_circle(pca, feature_names)

In [ ]:
plt.figure(figsize=(12, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=60, alpha=0.7)

for label, x, y in zip(labels, X_pca[:, 0], X_pca[:, 1]):
    plt.annotate(str(int(label)), xy=(x, y), xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('Projection des startups dans le plan principal')
plt.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
plt.axvline(x=0, color='k', linestyle='-', linewidth=0.5)
plt.grid(True, alpha=0.3)
plt.show()

---
# II. Clustering
---

## 1. KMeans (3 clusters)

In [ ]:
clustering_kmeans, kmeans = apply_kmeans(X_scaled, n_clusters=3)

print("Clusters (3) :")
print("=" * 60)
for i in range(3):
    members = labels[clustering_kmeans == i]
    print(f"\nCluster {i} ({len(members)} éléments) :")
    print(f"  {list(members)}")

In [ ]:
plot_clustering(X_pca, clustering_kmeans, labels, 'KMeans (K=3) - Startups')

In [ ]:
print("Profil moyen de chaque cluster :")
for i in range(3):
    mask = clustering_kmeans == i
    mean_values = X[mask].mean(axis=0)
    print(f"\nCluster {i}:")
    for j, name in enumerate(feature_names):
        print(f"  {name}: {mean_values[j]:,.0f}")

## 2. AgglomerativeClustering

In [ ]:
clustering_single = apply_agglomerative(X_scaled, n_clusters=3, linkage='single')
plot_clustering(X_pca, clustering_single, labels, 'Agglomerative - Single Linkage')

In [ ]:
clustering_ward = apply_agglomerative(X_scaled, n_clusters=3, linkage='ward')
plot_clustering(X_pca, clustering_ward, labels, 'Agglomerative - Ward Linkage')

In [ ]:
clustering_average = apply_agglomerative(X_scaled, n_clusters=3, linkage='average')
plot_clustering(X_pca, clustering_average, labels, 'Agglomerative - Average Linkage')

## 3. Détermination du nombre optimal (Silhouette)

In [ ]:
scores, best_k = compute_silhouette_scores(X_scaled)

In [ ]:
plot_silhouette_scores(scores)

## 4. Comparaison des méthodes

In [ ]:
results = compare_methods(X_scaled, n_clusters=3)

## 5. Avantages et inconvénients

### Classification Hiérarchique (AgglomerativeClustering)

**Avantages :**
- Pas besoin de spécifier K à l'avance
- Produit une hiérarchie complète (dendrogramme)
- Résultats déterministes
- Peut découvrir des clusters de formes arbitraires (avec single linkage)

**Inconvénients :**
- Complexité O(n²) ou O(n³)
- Ne passe pas à l'échelle pour grands datasets
- Décisions de fusion irréversibles
- Single linkage : effet de chaînage

---

### Partitionnement (KMeans)

**Avantages :**
- Très rapide O(n×K×iterations)
- Scalable pour grands datasets
- Clusters compacts et bien séparés
- Simple à comprendre et implémenter

**Inconvénients :**
- Nécessite K à l'avance
- Sensible à l'initialisation (non-déterministe)
- Sensible aux outliers
- Assume des clusters sphériques

## 6. Approche Hybride

In [ ]:
clustering_hyb, centers = clustering_hybride(X_scaled, n_clusters=3)
score_hyb = metrics.silhouette_score(X_scaled, clustering_hyb, metric='euclidean')
print(f"Approche Hybride - Silhouette : {score_hyb:.4f}")

print("\nClusters Hybride (3) :")
for i in range(3):
    members = labels[clustering_hyb == i]
    print(f"Cluster {i} ({len(members)} éléments): {list(members)}")

In [ ]:
plot_clustering(X_pca, clustering_hyb, labels, 'Approche Hybride (Ward + KMeans)')

---
# Conclusion

1. **ACP** : 2 composantes capturent ~87% de la variance
2. **Interprétation** : PC1 = performance globale, PC2 = stratégie d'investissement
3. **Clustering** : 3 profils de startups identifiables
   - Haute performance (gros investissements, hauts bénéfices)
   - Performance moyenne
   - En développement (faibles investissements)
4. **Méthode recommandée** : Ward ou KMeans